# **Preparo do ambiente**

### Montando Drive

In [ ]:
# Montando Drive
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/TCC/

Mounted at /content/drive/
/content/drive/MyDrive/TCC


### Baixando dependências

In [ ]:
# Instalando bibliotecas
!pip install torch_geometric
!pip install setproctitle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.2 MB/s  0:00:00


In [1]:
# Imports
import os
import shutil
import subprocess
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.sparse import coo_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay, auc, mean_squared_error
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from scipy.sparse.csgraph import connected_components
from scipy.sparse import lil_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score

/dados/luiz.gontijo/TCC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Definindo funções

In [2]:
def bizoia_resultados(n: int, embedding_size: int):
  # bizoia n resultados e dá a média dos trem
  media = {'accuracy': 0, 'precision': 0, 'recall': 0}
  for i in range(n):
      print(f'--- Resultado {i} ---')
      with open(f'node_classification/Resultados/predicoes/predict{i}.pkl', 'rb') as f:
          data = pickle.load(f)

      preds = data['preds']
      labels = data['labels']
      nodes = data['nodes']

      resultado = pd.DataFrame.from_dict({'preds': data['preds'], 'labels': data['labels']})

      with open(f'node_classification/Resultados/embeddings/embedding{i}.pkl', 'rb') as f:
          embed = pickle.load(f)

      s = 0
      for e in embed:
          s += len(e)

      print('Nós embeddados:', s)
      print('Nós labelados:', len(labels))
      print('Porcentagem de nós embeddados:', s/embedding_size, end='\n\n')

      print(resultado['preds'].value_counts(), end='\n\n')
      print(resultado['labels'].value_counts(), end='\n\n')

      accuracy = accuracy_score(resultado['labels'], resultado['preds'])
      precision = precision_score(resultado['labels'], resultado['preds'], zero_division=0)
      recall = recall_score(resultado['labels'], resultado['preds'])

      print(f'Accuracy {i}:', accuracy)
      print(f'Precision {i}:', precision)
      print(f'Recall {i}:', recall)
      media['accuracy'] += accuracy
      media['precision'] += precision
      media['recall'] += recall
      print()
      print()

  print('--- Média dos resultados ---')
  print('Accuracy:', media['accuracy']/n)
  print('Precision:', media['precision']/n)
  print('Recall:', media['recall']/n)
  print()


In [3]:
def remove_onlysenders(dataset: pd.DataFrame):
  '''
    Somente aplicável em datasets provenientes do SAML-D. Reduz iterativamente o dataset removendo arestas cujo sender nunca foi receiver, até todos terem sido receivers.
  '''
  df = dataset
  iteration = 0
  while True:
    accounts = pd.concat([df['Sender_account'], df['Receiver_account']]).unique()
    accounts_set = set(accounts)

    senders = set(df['Sender_account'])
    receivers = set(df['Receiver_account'])

    #print(f'Quantos accounts pssuem arestas bidirecionais?   {len(senders.intersection(receivers))} ({len(senders.intersection(receivers))/len(accounts_set)*100}%)')
    #print(f'Quantos accounts foram só senders?              {len(senders - receivers)} ({len(senders - receivers)/len(accounts_set)*100}%)')
    #print(f'Quantos accounts foram só receivers?            {len(receivers - senders)} ({len(receivers - senders)/len(accounts_set)*100}%)')

    bad_transactions = df[df['Sender_account'].isin(senders - receivers) == True]

    #print('Iteração', iteration)
    #print(f'Total de transações no dataset:   {df.size}')
    #print(f'Quantidade de "transações ruins": {bad_transactions.size} ({bad_transactions.size/df.size * 100})')

    if(bad_transactions.size == 0):
      very_good_transactions = df
      break
    else:
      iteration += 1

    # Quantos receiver accounts são só receivers
    #df[df['Receiver_account'].isin(receivers - senders) == True].size/df.size * 100
    #print('Quantidade de receiver que são só receivers:', df[df['Receiver_account'].isin(receivers - senders) == True].size/df.size * 100)

    good_transactions = df[df['Sender_account'].isin(senders - receivers) == False]

    df = good_transactions
    #print()
    #print('='*50)
    #print()

  print('Última iteração:', iteration)
  print('Quantidade de nós restantes:', very_good_transactions['Receiver_account'].unique().size)
  print('Porcentagem de transações removidas:', very_good_transactions.size/dataset.size)
  return very_good_transactions

In [4]:
def calcula_metricas(y_test, y_pred):
    qnt_zeroes = (y_pred == 0).sum()
    qnt_ones = (y_pred == 1).sum()
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    return qnt_zeroes, qnt_ones, accuracy, precision, recall, f1

In [5]:
def print_metricas(nome_classificador, qnt_zeroes, qnt_ones, accuracy, precision, recall, f1):
    print(f'{nome_classificador}:')
    print(f'Elementos 0: {qnt_zeroes}')
    print(f'Elementos 1: {qnt_ones}')
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1-score: {f1}')

# Leitura dos dados

In [9]:
!ls

'Ações do GitHub.ipynb'		 OpenGraph.ipynb    link_prediction
'Análise dos Resultados.ipynb'	 README.md	    node_classification
 History			 datasets	    requirements.txt
 LICENSE			 graph_generation   resultados_df.csv
 Models				 imgs


In [8]:
%cd OpenGraph

[Errno 2] No such file or directory: 'OpenGraph'
/dados/luiz.gontijo/TCC/OpenGraph


In [11]:
resultados_df = pd.read_csv('resultados_df.csv')

# **Análise dos Resultados**

O dataset **resultados_df** contém as métricas de desempenho do classificador de nós do OpenGraph, Random Forest e XGBoost, todos usando a embedding gerada pelo OpenGraph.

In [12]:
resultados_df

,modelo,dataset,adj_file,mask,qnt_zeroes,qnt_ones,accuracy,precision,recall,f1_score
0,OpenGraph,saml_d,original,40_30_30,256638,0,0.997576,0.000000,0.000000,0.000000
1,Random Forest,saml_d,original,40_30_30,153983,0,0.997578,0.000000,0.000000,0.000000
2,XGBoost,saml_d,original,40_30_30,153983,0,0.997578,0.000000,0.000000,0.000000
3,OpenGraph,saml_d,bi,40_30_30,94652,161986,0.369758,0.002667,0.694534,0.005313
4,Random Forest,saml_d,bi,40_30_30,153694,289,0.999441,0.996540,0.772118,0.870091
5,XGBoost,saml_d,bi,40_30_30,153789,194,0.998838,1.000000,0.520107,0.684303
6,OpenGraph,saml_d,conexo,40_30_30,256638,0,0.997576,0.000000,0.000000,0.000000
7,Random Forest,saml_d,conexo,40_30_30,153983,0,0.997578,0.000000,0.000000,0.000000
8,XGBoost,saml_d,conexo,40_30_30,153983,0,0.997578,0.000000,0.000000,0.000000
9,OpenGraph,saml_d,bi_conexo,40_30_30,102005,154633,0.398082,0.002522,0.627010,0.005024


# **Leitura dos Datasets**

#### Carregamento dos tipos de lavagem
Tipos de lavagem carregados por fora porque alguns podem estar ausentes em alguns dos recortes do SAML-D

In [ ]:
laundering_types = [
    # --- Tipos Normais (Simulações de comportamento lícito) ---
    'Normal_Cash_Deposits',
    'Normal_Cash_Withdrawal',
    'Normal_Fan_In',
    'Normal_Fan_Out',
    'Normal_Foward',
    'Normal_Group',
    'Normal_Mutual',
    'Normal_Periodical',
    'Normal_Plus_Mutual',
    'Normal_Small_Fan_Out',
    'Normal_single_large',

    # --- Tipos de Lavagem (Tipologias e Técnicas Suspeitas) ---
    'Behavioural_Change_1',
    'Behavioural_Change_2',
    'Bipartite',
    'Cash_Withdrawal',  # Classificado como anômalo neste dataset
    'Cycle',
    'Deposit-Send',
    'Fan_In',           # Padrão de agrupamento suspeito
    'Fan_Out',          # Padrão de dispersão suspeito
    'Gather-Scatter',
    'Layered_Fan_In',
    'Layered_Fan_Out',
    'Over-Invoicing',
    'Scatter-Gather',
    'Single_large',
    'Smurfing',
    'Stacked Bipartite',
    'Structuring'
]

#### **Dataset SAML-D e recortes**

##### SAML-D (original)
Dataset original, sem recortes. Geralmente pesado demais para os recursos disponíveis no Google Drive.

> * Nós: 855.460
> * Lavadores: 7.902 (0,92%)
> * Transações: 9.504.852

In [ ]:
# SAML-D original (855.460 nós, 7.902 lavadores, 9.504.852 transações)
saml_d = pd.read_csv('../Datasets/SAML-D/SAML-D.csv')

In [ ]:
df = saml_d
nome_dataset = 'saml_d'

##### SAML-D mini
Primeiros 11% das transações do SAML-D original. Recorte simples feito de maneira bruta, possivelmente com perdas topológicas (estruturas cortadas pela metade)

> * Nós: 325.888
> * Lavadores: 935 (0,29%)
> * Transações: 1.048.575

In [ ]:
# SAML-D mini (325.888 nós, 935 lavadores, 1.048.575 transações)
saml_d_mini = pd.read_csv('../Datasets/SAML-D/SAML-D_mini.csv')

In [ ]:
df = saml_d_mini
nome_dataset = 'saml_d_mini'

##### SAML-D sem nós exclusivamente remetentes
Foi realizada a remoção iterativa dos nós exclusivamente remetentes, junto com as arestas que neles incidiam, até que não houvesse mais nenhum.

In [ ]:
# SAML-D sem onlysenders (606.260 nós, 6.779 lavadores, 6.678.887 transações)
saml_d_nonlysenders = pd.read_csv('../Datasets/SAML-D/SAML-D_nonlysenders.csv')

In [ ]:
df = saml_d_nonlysenders
nome_dataset = 'saml_d_nonlysenders'

##### Metade do SAML-D sem nós exclusivamente remetentes
O algoritmo de remoção de nós exclusivamente remetentes foi aplicado ao SAML-D brutamente cortado pela metade.

In [ ]:
# SAML-D 50% sem onlysenders (443.381 nós, 2.948 lavadores, 2.699.819 transações)
saml_d_50_nonlysenders = pd.read_csv('../Datasets/SAML-D/SAML-D_50_nonlysenders.csv')

In [ ]:
df = saml_d_50_nonlysenders
nome_dataset = 'saml_d_50_nonlysenders'

##### Subgrafo Bidirecional do SAML-D

Foram retirados iterativamente nós que não possuíam cardinalidade de entrada, juntamente com suas arestas, até que não houvessem mais nós exclusivamente remetentes. Em seguida, o mesmo foi feito para nós que não possuíam cardinalidade de saída, até que não houvessem mais nós exclusivamente destinatários. O que sobra é o maior subgrafo bidirecional do SAML-D. Ainda há perda de sentido.

>* Nós: 66.370
>* Lavadores: 5.120 (7,71%)
>* Transações: 783.735

In [ ]:
# SAML-D subgrafo bidirecional (66.370 nós, 5.120 lavadores, 783.735 transações)
saml_d_intersection = pd.read_csv('../Datasets/SAML-D/SAML-D_intersection.csv')

In [ ]:
df = saml_d_intersection
nome_dataset = 'saml_d_intersection'

### **Tratamento do dataset lido**

#### Conversão das colunas

In [ ]:
# Convertendo Date e Time em Timestamp Unix
df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time']).astype('int64')

###### Criando mapeamento numérico para labels em string

In [ ]:
# Obtendo tipos de agrupamento
currencies          = df.groupby('Payment_currency')['Payment_currency'].count().sort_values(ascending=False).keys()
bank_locations      = df.groupby('Sender_bank_location')['Sender_bank_location'].count().sort_values(ascending=False).keys()
payment_types       = df.groupby('Payment_type')['Payment_type'].count().sort_values(ascending=False).keys()
laundering_types    = df.groupby('Laundering_type')['Laundering_type'].count().sort_values(ascending=False).keys()
accounts            = pd.concat([df['Sender_account'], df['Receiver_account']]).unique()
launderers          = pd.concat([
                                  df[df['Is_laundering'] == 1]['Sender_account'],
                                  df[df['Is_laundering'] == 1]['Receiver_account']
                                ]).unique()
is_launderer        = pd.Series(accounts).isin(launderers).astype(int)

# Enumeração dos tipos agrupados
account_map         = {account:i for i, account in enumerate(accounts)}
currency_map        = {currencies[i]:i for i in range(len(currencies))}
bank_location_map   = {bank_locations[i]:i for i in range(len(bank_locations))}
payment_type_map    = {payment_types[i]:i for i in range(len(payment_types))}
laundering_type_map = {laundering_types[i]:i for i in range(len(laundering_types))}

###### Dataset auxiliar de labels

Dataset auxiliar com a classificação de cada conta

In [ ]:
# Dataset auxiliar
Accounts = pd.DataFrame({
    'Accounts': accounts,
    'Launderers': pd.Series(accounts).isin(launderers).astype(int)
})

In [ ]:
print(f'{'Accounts:':<12}{Accounts.shape[0]:>10}')
print(f'{'Lavadores:':<12}{ launderers.size:>10}')
print(f'{'Transações:':<12}{    df.shape[0]:>10}')

Accounts:        66370
Lavadores:        5120
Transações:     783735


#### **Preparando variáveis de entrada**

##### Matriz de adjacência (COO Matrix)

###### Matriz original (adj_original)

In [ ]:
# Convertendo dataframe em matriz esparsa

row = df['Sender_account'].map(account_map)
col = df['Receiver_account'].map(account_map)

values = np.ones(df.shape[0])

num_nodes = Accounts.shape[0]

adj_original = coo_matrix((values, (row, col)), shape=(num_nodes, num_nodes))
adj = adj_original

###### Matriz bidirecional (adj_bi)
Transforma o grafo original em um grafo bidirecional

In [ ]:
# Transformação do grafo para grafo bidirecional
adj_bi = adj + adj.T
adj_bi.data = np.ones_like(adj_bi.data)

###### Matriz do grafo manualmente conexo (adj_conexo)

Foram formadas pequenas pontes sobre as diversas "ilhas" desconexas de nós para garantir message passing sobre todo o grafo

In [ ]:
adj = adj_original
# Obtendo componentes conexas
n_components, labels_cc = connected_components(adj)
print('Número de componentes em adj_original:', n_components )

# Lista de componentes (os nós que compõem a componente i)
components = [[] for _ in range(n_components)]
for node_index, component_label in enumerate(labels_cc):
    components[component_label].append(node_index)

adj_lil = adj.tolil() # Formato que permite manipulação do coo_sparse

# Conectando (bidirecionalmente) componentes (peso desbalanceado, estudar)
for i in range(-1, n_components-1):
  adj_lil[ components[i][0], components[i+1][0] ] = 1.0
  adj_lil[ components[i+1][0], components[i][0] ] = 1.0

adj_conexo = adj_lil.tocoo()
print('Número de componentes em adj_conexo:', connected_components(adj_conexo)[0] )

In [ ]:
adj = adj_bi
# Obtendo componentes conexas
n_components, labels_cc = connected_components(adj)
print('Número de componentes em adj_bi:', n_components )

# Lista de componentes (os nós que compõem a componente i)
components = [[] for _ in range(n_components)]
for node_index, component_label in enumerate(labels_cc):
    components[component_label].append(node_index)

adj_lil = adj.tolil() # Formato que permite manipulação do coo_sparse

# Conectando (bidirecionalmente) componentes (peso desbalanceado, estudar)
for i in range(-1, n_components-1):
  adj_lil[ components[i][0], components[i+1][0] ] = 1.0
  adj_lil[ components[i+1][0], components[i][0] ] = 1.0

adj_bi_conexo = adj_lil.tocoo()
print('Número de componentes em adj_bi_conexo:', connected_components(adj_bi_conexo)[0] )

Número de componentes em adj_bi: 14959
Número de componentes em adj_bi_conexo: 1


##### Vetor de Features

In [ ]:
# Features dos nós
feats = Accounts.drop(columns=['Accounts', 'Launderers']).values

##### Vetor de Labels

In [ ]:
# Labels dos nós (fraudadores ou não-fraudadores)
labels = np.array(is_launderer)

##### Vetor de Mascaramento

In [ ]:
# Configurando mascaramento dos nós

train_mask = np.zeros(num_nodes, dtype=bool)
val_mask = np.zeros(num_nodes, dtype=bool)
test_mask = np.zeros(num_nodes, dtype=bool)

# Divisórias percentuais de mascaramento
porcentagem_treino      = 0.4
porcentagem_teste       = 0.3
#validacao = 1 - (treino + teste), calculado automaticamente

div1 = num_nodes*porcentagem_treino
div2 = num_nodes*porcentagem_teste + div1
train_mask[:int(div1)] = True
val_mask[int(div1):int(div2)] = True
test_mask[int(div2):] = True


#### **Gravando variáveis de entrada**

In [ ]:
# Cria pasta para armazenar as variáveis, caso não exista
if not os.path.exists(f'datasets/{nome_dataset}'):
    os.makedirs(f'datasets/{nome_dataset}')

##### Matriz de adjacência

In [ ]:
# Grafo original
pickle.dump(adj_original, open(f'datasets/{nome_dataset}/adj_original.pkl', 'wb'))

In [ ]:
# Grafo bidirecionado
pickle.dump(adj_bi, open(f'datasets/{nome_dataset}/adj_bi.pkl', 'wb'))

In [ ]:
# Grafo conexo gerado diretamente a partir do original
pickle.dump(adj_conexo, open(f'datasets/{nome_dataset}/adj_conexo.pkl', 'wb'))

In [ ]:
# Grafo conexo gerado a partir do grafo  bidirecionado
pickle.dump(adj_bi_conexo, open(f'datasets/{nome_dataset}/adj_bi_conexo.pkl', 'wb'))

##### Vetor de Features

In [ ]:
pickle.dump(feats, open(f'datasets/{nome_dataset}/feats.pkl', 'wb'))

##### Vetor de Labels

In [ ]:
pickle.dump(labels, open(f'datasets/{nome_dataset}/label.pkl', 'wb'))

##### Vetor de Mascaramento

In [ ]:
nome_mask = f'{int(porcentagem_treino*100)}_{int((1-porcentagem_treino-porcentagem_teste)*100)}_{int(porcentagem_teste*100)}'
pickle.dump({
    'train': train_mask,
    'valid': val_mask,
    'test': test_mask
}, open(f'datasets/{nome_dataset}/mask_{nome_mask}.pkl', 'wb'))

# Análise dos Resultados